# TC-DrugRAG Demo

This notebook demonstrates the core functionality of TC-DrugRAG:
1. Building a Temporal Causal Knowledge Graph
2. Causal Path Retrieval
3. Hypothesis Generation with Uncertainty
4. Validation and Feedback Loop

In [ ]:
# Setup path
import sys
sys.path.insert(0, '..')

# Import modules
from datetime import datetime
from tc_drugrag.knowledge_graph import (
    TemporalCausalKG,
    Node,
    TemporalEdge,
    NodeType,
    CausalRelationType,
    EvidenceType,
)

## 1. Building a Temporal Causal Knowledge Graph

In [ ]:
# Initialize knowledge graph
kg = TemporalCausalKG(backend="networkx")

# Create nodes
metformin = Node(
    id="DB00331",
    name="Metformin",
    node_type=NodeType.DRUG,
    properties={"smiles": "CN(C)C(=N)NC(=N)N"},
)

ampk = Node(
    id="PRKAA1",
    name="AMPK",
    node_type=NodeType.PROTEIN,
)

mtor = Node(
    id="MTOR",
    name="mTOR",
    node_type=NodeType.PROTEIN,
)

glioblastoma = Node(
    id="DOID:3068",
    name="Glioblastoma",
    node_type=NodeType.DISEASE,
)

# Add nodes
for node in [metformin, ampk, mtor, glioblastoma]:
    kg.add_node(node)

print(f"Nodes added: {kg.num_nodes}")

In [ ]:
# Create temporal causal edges
edges = [
    TemporalEdge(
        source=metformin,
        target=ampk,
        relation_type=CausalRelationType.ACTIVATES,
        confidence=0.95,
        t_start=datetime(2001, 1, 1),
        evidence_type=EvidenceType.RANDOMIZED_CONTROLLED_TRIAL,
        evidence_sources=["PMID:11159568", "PMID:12615897"],
    ),
    TemporalEdge(
        source=ampk,
        target=mtor,
        relation_type=CausalRelationType.INHIBITS,
        confidence=0.90,
        t_start=datetime(2003, 1, 1),
        evidence_type=EvidenceType.PRECLINICAL_IN_VITRO,
        evidence_sources=["PMID:14517290"],
    ),
    TemporalEdge(
        source=mtor,
        target=glioblastoma,
        relation_type=CausalRelationType.ASSOCIATED_WITH,
        confidence=0.85,
        t_start=datetime(2008, 1, 1),
        evidence_type=EvidenceType.CLINICAL_OBSERVATIONAL,
        evidence_sources=["PMID:18669468", "PMID:19139180"],
    ),
]

for edge in edges:
    kg.add_edge(edge)

print(f"Edges added: {kg.num_edges}")
print(f"\nGraph statistics: {kg.get_statistics()}")

## 2. Causal Path Retrieval

In [ ]:
# Find causal paths from Metformin to Glioblastoma
paths = list(kg.find_causal_paths(
    source_id="DB00331",
    target_id="DOID:3068",
    max_depth=4,
))

print(f"Found {len(paths)} causal path(s)")

for i, path in enumerate(paths):
    print(f"\nPath {i+1}:")
    print(f"  Mechanism: {path.get_mechanism_description()}")
    print(f"  Length: {path.length}")
    print(f"  Confidence: {path.get_path_confidence():.3f}")
    print(f"  Temporally consistent: {path.is_temporally_consistent()}")

In [ ]:
# Score the path
from tc_drugrag.retrieval import CausalPathScorer

scorer = CausalPathScorer()

for path in paths:
    breakdown = scorer.get_score_breakdown(path)
    print("\nScore Breakdown:")
    for key, value in breakdown.items():
        print(f"  {key}: {value:.3f}" if isinstance(value, float) else f"  {key}: {value}")

## 3. Hypothesis Generation with Uncertainty

In [ ]:
from tc_drugrag.hypothesis import DrugDiscoveryHypothesis, UncertaintyQuantifier

# Create a hypothesis (normally done by LLM)
hypothesis = DrugDiscoveryHypothesis(
    drug="Metformin",
    target="AMPK",
    disease="Glioblastoma",
    mechanism="Metformin activates AMPK, which inhibits mTOR signaling, "
              "leading to reduced cell proliferation in glioblastoma.",
    confidence=0.73,
    confidence_interval=(0.61, 0.85),
    novelty_score=0.65,
    evidence_summary=[
        "Metformin is a known AMPK activator (PMID:11159568)",
        "AMPK inhibits mTOR signaling (PMID:14517290)",
        "mTOR is hyperactive in glioblastoma (PMID:18669468)",
    ],
    suggested_experiments=[
        "Test metformin in glioblastoma cell lines (U87, U251)",
        "Measure AMPK/mTOR pathway activation",
        "Assess combination with temozolomide",
    ],
)

print(hypothesis.get_summary())

In [ ]:
# Uncertainty decomposition
uq = UncertaintyQuantifier()
decomposition = uq.get_uncertainty_decomposition(paths)

print("Uncertainty Decomposition:")
print(f"  Aleatoric (irreducible): {decomposition['aleatoric']:.3f}")
print(f"  Epistemic (reducible): {decomposition['epistemic']:.3f}")
print(f"  Total: {decomposition['total']:.3f}")
print("\nSources:")
for source, value in decomposition['sources'].items():
    print(f"  {source}: {value:.3f}")

## 4. Validation

In [ ]:
from tc_drugrag.validation import ADMETPredictor

# Predict ADMET properties
predictor = ADMETPredictor(use_api=False)
admet = predictor.predict(drug="Metformin")

print("ADMET Properties:")
for prop, value in admet.items():
    if isinstance(value, float):
        print(f"  {prop}: {value:.3f}")

## Summary

TC-DrugRAG provides:
1. **Temporal causal edges** - relationships with time validity and confidence
2. **Causal path retrieval** - find mechanistic paths with scoring
3. **Uncertainty quantification** - calibrated confidence intervals
4. **In-silico validation** - docking, ADMET, feedback loop

This enables hypothesis-driven drug discovery with interpretable mechanisms and honest uncertainty estimates.